<a href="https://colab.research.google.com/github/ximecaceres/materia-IA-/blob/main/ecommerce_cosmeticos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agente 1 para normalizar dataset


## SECCIÓN 1: CARGAR EL DATASET E IMPORTAR LIBRERÍAS

In [18]:
# Importamos la librería pandas, esencial para trabajar con DataFrames.
import pandas as pd

# Definimos la ruta del archivo CSV.
# Se corrige el nombre del archivo con un espacio extra, según lo que aparece en el sistema de archivos.
file_path = '/content/E-commerce  cosmetic dataset.csv'

# Intentamos cargar el archivo CSV utilizando la codificación UTF-8 como primera opción.
# Si hay un error de codificación, se puede intentar con 'latin1' o 'ISO-8859-1'.
try:
    df = pd.read_csv(file_path, encoding='utf-8')
    print("Dataset cargado exitosamente con codificación 'utf-8'.")
except UnicodeDecodeError:
    print("Error de codificación con 'utf-8'. Intentando con 'latin1'...")
    try:
        df = pd.read_csv(file_path, encoding='latin1')
        print("Dataset cargado exitosamente con codificación 'latin1'.")
    except Exception as e:
        print(f"No se pudo cargar el dataset con 'latin1' tampoco. Error: {e}")
        print("Por favor, verifique la codificación de su archivo CSV.")
        df = None # Aseguramos que df sea None si la carga falla

# Si el DataFrame se cargó correctamente, mostramos las primeras filas para una verificación inicial.
if df is not None:
    print("\nPrimeras 5 filas del dataset original:")
    display(df.head())

Error de codificación con 'utf-8'. Intentando con 'latin1'...
Dataset cargado exitosamente con codificación 'latin1'.

Primeras 5 filas del dataset original:


,product_name,website,country,category,subcategory,title-href,price,brand,ingredients,form,type,color,size,rating,noofratings
0,"Carlton London Incense Eau da parfum, Premium ...",Flipkart,India,body,perfume,https://www.amazon.in/Carlton-London-Limited-I...,599.0,Carlton London,NaN,aerosol,NaN,"Top Note: Orange Blossom, Blackberry | Heart N...",100,3.9,19
1,CHARLENE SPRAY MIST PERFUME 30 - INTIMATE (PAC...,Flipkart,India,body,perfume,https://www.amazon.in/CHARLENE-SPRAY-MIST-PERF...,149.0,Charlene,NaN,aerosol,NaN,Unit count type:,30,4.4,"4,031"
2,CHARLENE SPRAY MIST PERFUME 30 - INTIMATE (PAC...,Flipkart,India,body,perfume,https://www.amazon.in/CHARLENE-SPRAY-MIST-PERF...,298.0,Charlene,NaN,aerosol,NaN,Unit count type:,30,4.4,"4,072"
3,DENVER Black Code Perfume - 60 | Eau de Parfum...,Flipkart,India,body,perfume,https://www.amazon.in/DENVER-Black-Code-Perfum...,245.0,Denver,NaN,aerosol,NaN,Long-Lasting Scent,60,4.2,61
4,Denver Hamilton Perfume - 100 | Long Lasting P...,Flipkart,India,body,perfume,https://www.amazon.in/Denver-Perfume-Hamilton-...,422.0,Denver,NaN,aerosol,NaN,Long-Lasting Scent,100,4.3,342


In [19]:
print('Re-ejecutando la celda de carga de dataset con la ruta corregida.')

Re-ejecutando la celda de carga de dataset con la ruta corregida.


## SECCIÓN 2: NORMALIZACIÓN Y LIMPIEZA PASO A PASO

In [20]:
# 1. Normalizar nombres de columnas (snake_case)

# Función para convertir un string a snake_case
def to_snake_case(name):
    # Convierte a minúsculas
    name = name.lower()
    # Reemplaza espacios, guiones y otros caracteres no alfanuméricos con guiones bajos
    # Usa regex para ser más robusto
    name = pd.Series(name).str.replace(r'[^a-z0-9_]+', '_', regex=True).iloc[0]
    # Elimina guiones bajos al inicio o al final y reemplaza múltiples guiones bajos por uno solo
    name = pd.Series(name).str.strip('_').str.replace(r'__+', '_', regex=True).iloc[0]
    return name

# Aplicamos la función a todos los nombres de columnas del DataFrame
if df is not None:
    original_columns = df.columns.tolist()
    df.columns = [to_snake_case(col) for col in df.columns]
    print("\nNombres de columnas normalizados:")
    for original, new in zip(original_columns, df.columns):
        print(f"  '{original}' -> '{new}'")


Nombres de columnas normalizados:
  'product_name' -> 'product_name'
  'website' -> 'website'
  'country' -> 'country'
  'category' -> 'category'
  'subcategory' -> 'subcategory'
  'title-href' -> 'title_href'
  'price' -> 'price'
  'brand' -> 'brand'
  'ingredients' -> 'ingredients'
  'form' -> 'form'
  'type' -> 'type'
  'color' -> 'color'
  'size' -> 'size'
  'rating' -> 'rating'
  'noofratings' -> 'noofratings'


In [21]:
print('Re-ejecutando la celda de normalización de columnas.')

Re-ejecutando la celda de normalización de columnas.


In [22]:
# 2. Verificar valores nulos

# Calculamos el número de valores nulos por columna
if df is not None:
    null_counts = df.isnull().sum()
    print("\nValores nulos por columna (antes de la limpieza):")
    display(null_counts[null_counts > 0].to_frame(name='Nulos'))
    if null_counts.sum() == 0:
        print("No hay valores nulos en el dataset.")


Valores nulos por columna (antes de la limpieza):


,Nulos
price,317
ingredients,6015
type,2681
color,1989
size,3166
rating,2067
noofratings,459


In [23]:
print('Re-ejecutando la celda de verificación de nulos.')

Re-ejecutando la celda de verificación de nulos.


In [24]:
# 3. Reemplazar valores nulos

if df is not None:
    for col in df.columns:
        # Si la columna es numérica, rellenamos los nulos con 0
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(0)
            print(f"  Columna '{col}': valores nulos rellenados con 0.")
        # Si la columna es de tipo objeto (generalmente texto), rellenamos con 'no_definido'
        elif pd.api.types.is_object_dtype(df[col]):
            df[col] = df[col].fillna('no_definido')
            print(f"  Columna '{col}': valores nulos rellenados con 'no_definido'.")

    print("\nValores nulos después de rellenar:")
    display(df.isnull().sum().to_frame(name='Nulos'))

  Columna 'product_name': valores nulos rellenados con 'no_definido'.
  Columna 'website': valores nulos rellenados con 'no_definido'.
  Columna 'country': valores nulos rellenados con 'no_definido'.
  Columna 'category': valores nulos rellenados con 'no_definido'.
  Columna 'subcategory': valores nulos rellenados con 'no_definido'.
  Columna 'title_href': valores nulos rellenados con 'no_definido'.
  Columna 'price': valores nulos rellenados con 0.
  Columna 'brand': valores nulos rellenados con 'no_definido'.
  Columna 'ingredients': valores nulos rellenados con 'no_definido'.
  Columna 'form': valores nulos rellenados con 'no_definido'.
  Columna 'type': valores nulos rellenados con 'no_definido'.
  Columna 'color': valores nulos rellenados con 'no_definido'.
  Columna 'size': valores nulos rellenados con 'no_definido'.
  Columna 'rating': valores nulos rellenados con 'no_definido'.
  Columna 'noofratings': valores nulos rellenados con 'no_definido'.

Valores nulos después de rellen

,Nulos
product_name,0
website,0
country,0
category,0
subcategory,0
title_href,0
price,0
brand,0
ingredients,0
form,0


In [25]:
print('Re-ejecutando la celda de reemplazo de nulos.')

Re-ejecutando la celda de reemplazo de nulos.


In [26]:
# 4. Limpieza de texto (caracteres especiales rotos o extraños)

# Esta función limpia caracteres no ASCII y normaliza espacios en cadenas de texto.
def clean_text(text):
    if isinstance(text, str):
        # Elimina caracteres no ASCII (pueden ser 'rotos' o 'extraños')
        text = text.encode('ascii', 'ignore').decode('ascii')
        # Reemplaza múltiples espacios por uno solo
        text = ' '.join(text.split())
        # Elimina espacios al inicio y al final
        text = text.strip()
    return text

if df is not None:
    # Identificamos las columnas de tipo 'object' (que suelen contener texto)
    text_columns = df.select_dtypes(include='object').columns

    # Aplicamos la función de limpieza a cada columna de texto
    for col in text_columns:
        df[col] = df[col].apply(clean_text)
        print(f"  Columna '{col}': texto limpiado.")

    print("\nEjemplo de limpieza de texto (primeras filas con columnas de texto):")
    display(df[text_columns].head())

  Columna 'product_name': texto limpiado.
  Columna 'website': texto limpiado.
  Columna 'country': texto limpiado.
  Columna 'category': texto limpiado.
  Columna 'subcategory': texto limpiado.
  Columna 'title_href': texto limpiado.
  Columna 'brand': texto limpiado.
  Columna 'ingredients': texto limpiado.
  Columna 'form': texto limpiado.
  Columna 'type': texto limpiado.
  Columna 'color': texto limpiado.
  Columna 'size': texto limpiado.
  Columna 'rating': texto limpiado.
  Columna 'noofratings': texto limpiado.

Ejemplo de limpieza de texto (primeras filas con columnas de texto):


,product_name,website,country,category,subcategory,title_href,brand,ingredients,form,type,color,size,rating,noofratings
0,"Carlton London Incense Eau da parfum, Premium ...",Flipkart,India,body,perfume,https://www.amazon.in/Carlton-London-Limited-I...,Carlton London,no_definido,aerosol,no_definido,"Top Note: Orange Blossom, Blackberry | Heart N...",100,3.9,19
1,CHARLENE SPRAY MIST PERFUME 30 - INTIMATE (PAC...,Flipkart,India,body,perfume,https://www.amazon.in/CHARLENE-SPRAY-MIST-PERF...,Charlene,no_definido,aerosol,no_definido,Unit count type:,30,4.4,"4,031"
2,CHARLENE SPRAY MIST PERFUME 30 - INTIMATE (PAC...,Flipkart,India,body,perfume,https://www.amazon.in/CHARLENE-SPRAY-MIST-PERF...,Charlene,no_definido,aerosol,no_definido,Unit count type:,30,4.4,"4,072"
3,DENVER Black Code Perfume - 60 | Eau de Parfum...,Flipkart,India,body,perfume,https://www.amazon.in/DENVER-Black-Code-Perfum...,Denver,no_definido,aerosol,no_definido,Long-Lasting Scent,60,4.2,61
4,Denver Hamilton Perfume - 100 | Long Lasting P...,Flipkart,India,body,perfume,https://www.amazon.in/Denver-Perfume-Hamilton-...,Denver,no_definido,aerosol,no_definido,Long-Lasting Scent,100,4.3,342


In [27]:
print('Re-ejecutando la celda de limpieza de texto.')

Re-ejecutando la celda de limpieza de texto.


In [28]:
# 5. Mostrar estructura final

if df is not None:
    print(f"\nNúmero total de filas: {df.shape[0]}")
    print(f"Número total de columnas: {df.shape[1]}")
    print("\nVista previa (head) del DataFrame final después de la limpieza:")
    display(df.head())


Número total de filas: 12615
Número total de columnas: 15

Vista previa (head) del DataFrame final después de la limpieza:


,product_name,website,country,category,subcategory,title_href,price,brand,ingredients,form,type,color,size,rating,noofratings
0,"Carlton London Incense Eau da parfum, Premium ...",Flipkart,India,body,perfume,https://www.amazon.in/Carlton-London-Limited-I...,599.0,Carlton London,no_definido,aerosol,no_definido,"Top Note: Orange Blossom, Blackberry | Heart N...",100,3.9,19
1,CHARLENE SPRAY MIST PERFUME 30 - INTIMATE (PAC...,Flipkart,India,body,perfume,https://www.amazon.in/CHARLENE-SPRAY-MIST-PERF...,149.0,Charlene,no_definido,aerosol,no_definido,Unit count type:,30,4.4,"4,031"
2,CHARLENE SPRAY MIST PERFUME 30 - INTIMATE (PAC...,Flipkart,India,body,perfume,https://www.amazon.in/CHARLENE-SPRAY-MIST-PERF...,298.0,Charlene,no_definido,aerosol,no_definido,Unit count type:,30,4.4,"4,072"
3,DENVER Black Code Perfume - 60 | Eau de Parfum...,Flipkart,India,body,perfume,https://www.amazon.in/DENVER-Black-Code-Perfum...,245.0,Denver,no_definido,aerosol,no_definido,Long-Lasting Scent,60,4.2,61
4,Denver Hamilton Perfume - 100 | Long Lasting P...,Flipkart,India,body,perfume,https://www.amazon.in/Denver-Perfume-Hamilton-...,422.0,Denver,no_definido,aerosol,no_definido,Long-Lasting Scent,100,4.3,342


In [29]:
print('Re-ejecutando la celda de mostrar estructura final.')

Re-ejecutando la celda de mostrar estructura final.


## SECCIÓN 3: GUARDAR EL RESULTADO

In [30]:
# Definimos el nombre del nuevo archivo CSV limpio
output_file_path = 'cosmetics_limpio.csv'

if df is not None:
    # Exportamos el DataFrame limpio a un nuevo archivo CSV
    # index=False evita que pandas escriba el índice del DataFrame como una columna en el CSV
    df.to_csv(output_file_path, index=False, encoding='utf-8')
    print(f"\nDataFrame limpio guardado exitosamente en '{output_file_path}'.")


DataFrame limpio guardado exitosamente en 'cosmetics_limpio.csv'.


In [31]:
print('Re-ejecutando la celda de guardar resultado.')

Re-ejecutando la celda de guardar resultado.


## VERIFICACIÓN DEL ARCHIVO GENERADO

In [32]:
# Cargamos el archivo recién guardado para verificar su integridad
try:
    df_cleaned_check = pd.read_csv(output_file_path, encoding='utf-8')
    print(f"\n'{output_file_path}' cargado para verificación.")

    # Verificamos que la suma total de valores nulos sea 0
    total_nulls_check = df_cleaned_check.isnull().sum().sum()

    print(f"Número total de valores nulos en '{output_file_path}': {total_nulls_check}")

    if total_nulls_check == 0:
        print("¡Confirmación exitosa! La suma de nulos en el archivo limpio es estrictamente 0.")
        print("Primeras 5 filas del archivo limpio verificado:")
        display(df_cleaned_check.head())
    else:
        print("Advertencia: Aún existen valores nulos en el archivo limpio. Por favor, revise el proceso de limpieza.")
        print("Valores nulos restantes por columna:")
        display(df_cleaned_check.isnull().sum()[df_cleaned_check.isnull().sum() > 0].to_frame(name='Nulos'))

except Exception as e:
    print(f"Error al cargar o verificar el archivo '{output_file_path}'. Error: {e}")


'cosmetics_limpio.csv' cargado para verificación.
Número total de valores nulos en 'cosmetics_limpio.csv': 0
¡Confirmación exitosa! La suma de nulos en el archivo limpio es estrictamente 0.
Primeras 5 filas del archivo limpio verificado:


,product_name,website,country,category,subcategory,title_href,price,brand,ingredients,form,type,color,size,rating,noofratings
0,"Carlton London Incense Eau da parfum, Premium ...",Flipkart,India,body,perfume,https://www.amazon.in/Carlton-London-Limited-I...,599.0,Carlton London,no_definido,aerosol,no_definido,"Top Note: Orange Blossom, Blackberry | Heart N...",100,3.9,19
1,CHARLENE SPRAY MIST PERFUME 30 - INTIMATE (PAC...,Flipkart,India,body,perfume,https://www.amazon.in/CHARLENE-SPRAY-MIST-PERF...,149.0,Charlene,no_definido,aerosol,no_definido,Unit count type:,30,4.4,"4,031"
2,CHARLENE SPRAY MIST PERFUME 30 - INTIMATE (PAC...,Flipkart,India,body,perfume,https://www.amazon.in/CHARLENE-SPRAY-MIST-PERF...,298.0,Charlene,no_definido,aerosol,no_definido,Unit count type:,30,4.4,"4,072"
3,DENVER Black Code Perfume - 60 | Eau de Parfum...,Flipkart,India,body,perfume,https://www.amazon.in/DENVER-Black-Code-Perfum...,245.0,Denver,no_definido,aerosol,no_definido,Long-Lasting Scent,60,4.2,61
4,Denver Hamilton Perfume - 100 | Long Lasting P...,Flipkart,India,body,perfume,https://www.amazon.in/Denver-Perfume-Hamilton-...,422.0,Denver,no_definido,aerosol,no_definido,Long-Lasting Scent,100,4.3,342


In [33]:
print('Re-ejecutando la celda de verificación final.')

Re-ejecutando la celda de verificación final.


# AGENTE 2: ENTRENADOR

Objetivo: Aplicar validación, entrenar, seleccionar el modelo y obtener métricas.


#### 1. Validación y Limpieza de la Variable Objetivo 'rating'

Se revisó la columna rating para asegurar que solo contenga calificaciones válidas. Los valores incorrectos, vacíos o fuera del rango permitido (0 a 5) fueron eliminados. De esta forma, el modelo se entrena únicamente con datos confiables, mejorando la calidad de las predicciones y de las métricas obtenidas.

In [34]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor # Añadido GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error # Añadido MAE
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

print("[Agente 2]: Iniciando fase de entrenamiento y seleccion de modelos...")

# 1. Carga de datos normalizados
df_cosmetics = pd.read_csv('cosmetics_limpio.csv')

# Correcciones y Validaciones para la variable 'rating'
print("\n[Agente 2]: Validando y limpiando la variable objetivo 'rating'...")

# Identificar y corregir valores con separador de miles (ej. '14,338') -> '14.338'
df_cosmetics['rating'] = df_cosmetics['rating'].astype(str).str.replace(',', '.', regex=False)

# Convertir 'rating' a numérico, forzando errores a NaN
df_cosmetics['rating'] = pd.to_numeric(df_cosmetics['rating'], errors='coerce')

# Contar valores NaN resultantes (de 'no_definido' o errores de conversión)
nan_ratings_count = df_cosmetics['rating'].isnull().sum()
if nan_ratings_count > 0:
    print(f"  - Se detectaron {nan_ratings_count} valores no numéricos o 'no_definido' en 'rating'.")
    print("    Estos registros serán eliminados para asegurar la calidad de la variable objetivo.")
    df_cosmetics.dropna(subset=['rating'], inplace=True)
    print(f"  - Total de filas después de eliminar NaNs en 'rating': {len(df_cosmetics)}")

# Contar valores fuera del rango 0-5
invalid_range_count = df_cosmetics[(df_cosmetics['rating'] < 0) | (df_cosmetics['rating'] > 5)].shape[0]
if invalid_range_count > 0:
    print(f"  - Se detectaron {invalid_range_count} valores en 'rating' fuera del rango [0-5].")
    print("    Estos registros serán eliminados para asegurar ratings válidos.")
    df_cosmetics = df_cosmetics[(df_cosmetics['rating'] >= 0) & (df_cosmetics['rating'] <= 5)]
    print(f"  - Total de filas después de eliminar ratings fuera de rango: {len(df_cosmetics)}")

# 2. Preparacion de caracteristicas (Features) y Target
# Se añade 'noofratings' como una característica, ya que es altamente relevante para predecir el rating.
# Se asegura que 'noofratings' sea numérico, rellenando NaNs con 0 (ya que es una característica, no el target).
# Se corrige el manejo inicial de 'noofratings' que podía haber sido 'no_definido' del Agente 1.
df_cosmetics['noofratings'] = pd.to_numeric(df_cosmetics['noofratings'], errors='coerce').fillna(0)

X = df_cosmetics[['brand', 'category', 'price', 'noofratings']].copy() # Se añade 'noofratings'
y = df_cosmetics['rating'].copy()

# Corrección: Codificacion de variables categoricas (One-Hot Encoding) ---
# Se reemplaza LabelEncoder por OneHotEncoder para 'brand' y 'category'.
# LabelEncoder asume una relación ordinal, lo cual no es adecuado para variables nominales como 'brand' y 'category'.
# OneHotEncoder crea nuevas columnas binarias, evitando esta asunción y mejorando el rendimiento del modelo.

categorical_features = ['brand', 'category']
numerical_features = ['price', 'noofratings'] # Se añade 'noofratings'

# Creamos un preprocesador para aplicar diferentes transformaciones a diferentes columnas
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_features), # Para características numéricas, no hacer nada
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features) # One-Hot Encoding para categóricas
    ])

# 4. Division del dataset (Train / Test Split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Datos segmentados: {X_train.shape[0]} muestras de entrenamiento y {X_test.shape[0]} de prueba.")

# 5. Evaluacion de modelos candidatos mediante Validacion Cruzada
print("\n[Agente 2]: Evaluando candidatos mediante Cross-Validation...")

# Candidato A: Regresion Lineal con Pipeline de preprocesamiento
pipeline_lr = Pipeline(steps=[('preprocessor', preprocessor),
                              ('regressor', LinearRegression())])
scores_a = cross_val_score(pipeline_lr, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
rmse_cv_a = np.sqrt(-scores_a.mean())
mae_cv_a = np.mean(np.abs(cross_val_score(pipeline_lr, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')))

# Candidato B: Random Forest Regressor con Pipeline de preprocesamiento
pipeline_rf = Pipeline(steps=[('preprocessor', preprocessor),
                              ('regressor', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))]) # n_jobs=-1 para paralelización
scores_b = cross_val_score(pipeline_rf, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
rmse_cv_b = np.sqrt(-scores_b.mean())
mae_cv_b = np.mean(np.abs(cross_val_score(pipeline_rf, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')))

# Candidato C: Gradient Boosting Regressor con Pipeline de preprocesamiento (añadido para mayor robustez)
pipeline_gb = Pipeline(steps=[('preprocessor', preprocessor),
                              ('regressor', GradientBoostingRegressor(n_estimators=100, random_state=42))])
scores_c = cross_val_score(pipeline_gb, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
rmse_cv_c = np.sqrt(-scores_c.mean())
mae_cv_c = np.mean(np.abs(cross_val_score(pipeline_gb, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')))

print(f"   -> RMSE Promedio (CV) - Regresion Lineal: {rmse_cv_a:.4f} | MAE Promedio (CV): {mae_cv_a:.4f}")
print(f"   -> RMSE Promedio (CV) - Random Forest: {rmse_cv_b:.4f} | MAE Promedio (CV): {mae_cv_b:.4f}")
print(f"   -> RMSE Promedio (CV) - Gradient Boosting: {rmse_cv_c:.4f} | MAE Promedio (CV): {mae_cv_c:.4f}")

# 6. Seleccion automatica del mejor modelo en base al menor error (RMSE)
model_performances = {
    "Linear Regression": {"model": pipeline_lr, "rmse": rmse_cv_a, "mae": mae_cv_a},
    "Random Forest Regressor": {"model": pipeline_rf, "rmse": rmse_cv_b, "mae": mae_cv_b},
    "Gradient Boosting Regressor": {"model": pipeline_gb, "rmse": rmse_cv_c, "mae": mae_cv_c}
}

mejor_modelo_info = min(model_performances.items(), key=lambda x: x[1]['rmse'])
nombre_modelo = mejor_modelo_info[0]
mejor_modelo = mejor_modelo_info[1]['model']

print(f"\n[Agente 2]: {nombre_modelo} seleccionado como el modelo óptimo debido a menor RMSE en Cross-Validation.")

# Ajuste final del modelo seleccionado con los datos de entrenamiento completos
mejor_modelo.fit(X_train, y_train)

# 7. Evaluacion final del modelo seleccionado en el conjunto de prueba
y_pred = mejor_modelo.predict(X_test)
mse_final = mean_squared_error(y_test, y_pred)
rmse_final = np.sqrt(mse_final)
mae_final = mean_absolute_error(y_test, y_pred) # Cálculo del MAE final
r2_final = r2_score(y_test, y_pred)

print("\nMetricas finales del modelo seleccionado en el conjunto de prueba:")
print(f"   - Error Cuadratico Medio (MSE): {mse_final:.4f}")
print(f"   - Raiz del Error Cuadratico Medio (RMSE): {rmse_final:.4f}")
print(f"   - Error Absoluto Medio (MAE): {mae_final:.4f}") # MAE añadido
print(f"   - Coeficiente de Determinacion (R2): {r2_final:.4f}")

# Conclusiones automáticas sobre el desempeño del modelo
print("\n[Agente 2]: Generando conclusiones automáticas...")
if r2_final < 0.1:
    conclusion_r2 = "El valor de R² es muy bajo, lo que indica que el modelo explica muy poca de la varianza en la variable objetivo. Es probable que las características actuales sean insuficientes o que la relación no sea lineal. Se recomienda una exploración de características más profunda o modelos más complejos."
elif r2_final < 0.4:
    conclusion_r2 = "El valor de R² es bajo, el modelo tiene un poder predictivo limitado. Podría haber variables predictoras importantes que no se están utilizando, o la relación subyacente es más compleja de lo que el modelo actual puede capturar."
elif r2_final < 0.7:
    conclusion_r2 = "El valor de R² es moderado, el modelo explica una parte razonable de la varianza. Con mejoras adicionales en características o ajuste de hiperparámetros, el rendimiento podría ser mejor."
else:
    conclusion_r2 = "El valor de R² es alto, lo que indica un buen ajuste del modelo a los datos. El modelo tiene un buen poder predictivo."

print(f"Conclusión sobre R²: {conclusion_r2}")
print(f"El RMSE final de {rmse_final:.2f} y MAE de {mae_final:.2f} indican el error promedio en las predicciones. Un MAE de {mae_final:.2f} significa que, en promedio, las predicciones del modelo se desvían de los valores reales en {mae_final:.2f} unidades de rating.")

# SECCION 5: EXPORTACION DE METRICAS Y MODELO

print("\n[Agente 2]: Exportando artefactos y reporte para el Agente 3...")

# Guardado del modelo y codificadores en archivos binarios
joblib.dump(mejor_modelo, 'modelo_cosmetics.pkl')

# NOTA: No es necesario guardar los LabelEncoders, ya que se usa OneHotEncoder dentro del Pipeline
# y el preprocesador se guarda como parte del pipeline_lr o pipeline_rf.
# joblib.dump(le_brand, 'encoder_brand.pkl')
# joblib.dump(le_category, 'encoder_category.pkl')

# Generacion del archivo de texto plano con las metricas para el RAG
reporte_metricas = f"""
REPORTE DE RENDIMIENTO DEL AGENTE 2 (ENTRENADOR)
------------------------------------------------
Modelo seleccionado: {nombre_modelo}
Métricas de evaluación del modelo:
- Error Cuadratico Medio (MSE): {mse_final:.4f}
- Raiz del Error Cuadratico Medio (RMSE): {rmse_final:.4f}
- Error Absoluto Medio (MAE): {mae_final:.4f}
- Coeficiente de Determinacion (R2): {r2_final:.4f}

Variables utilizadas para la prediccion: brand, category, price, noofratings
Variable objetivo predicha: rating

Conclusiones:
{conclusion_r2}
El MAE de {mae_final:.2f} significa que, en promedio, las predicciones del modelo se desvían de los valores reales en {mae_final:.2f} unidades de rating.
"""

with open('metricas_agente2.txt', 'w', encoding='utf-8') as f:
    f.write(reporte_metricas)

print("[Agente 2]: Fase completada de manera exitosa. Archivo 'modelo_cosmetics.pkl' y 'metricas_agente2.txt' generados.")

[Agente 2]: Iniciando fase de entrenamiento y seleccion de modelos...

[Agente 2]: Validando y limpiando la variable objetivo 'rating'...
  - Se detectaron 2081 valores no numéricos o 'no_definido' en 'rating'.
    Estos registros serán eliminados para asegurar la calidad de la variable objetivo.
  - Total de filas después de eliminar NaNs en 'rating': 10534
  - Se detectaron 51 valores en 'rating' fuera del rango [0-5].
    Estos registros serán eliminados para asegurar ratings válidos.
  - Total de filas después de eliminar ratings fuera de rango: 10483
Datos segmentados: 8386 muestras de entrenamiento y 2097 de prueba.

[Agente 2]: Evaluando candidatos mediante Cross-Validation...
   -> RMSE Promedio (CV) - Regresion Lineal: 0.4793 | MAE Promedio (CV): 0.3041
   -> RMSE Promedio (CV) - Random Forest: 0.4545 | MAE Promedio (CV): 0.2318
   -> RMSE Promedio (CV) - Gradient Boosting: 0.4423 | MAE Promedio (CV): 0.2676

[Agente 2]: Gradient Boosting Regressor seleccionado como el modelo 

AGENTE 3: COMUNICADOR


### Celda 1: Instalaciones e Importaciones Base
Instalamos las librerías necesarias para el ecosistema LangChain y la ejecución de modelos de Hugging Face en modo optimizado.

In [46]:
!pip install -q transformers accelerate sentence-transformers faiss-cpu langchain-community

import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)

from langchain_community.llms import HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

CELDA 2: Cargar el modelo Transformer

In [67]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=150,
    temperature=0.1,
    do_sample=False,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=pipe)

print("LLM cargado correctamente.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


LLM cargado correctamente.


### Celda 3: Configuración del Agente de Datos
Cargar dataset

In [59]:
# Carga del dataset limpio
df_cosmetics = pd.read_csv('cosmetics_limpio.csv')

# Limpieza de la columna rating

df_cosmetics['rating'] = (
    df_cosmetics['rating']
    .astype(str)
    .str.replace(',', '.', regex=False)
)

df_cosmetics['rating'] = pd.to_numeric(
    df_cosmetics['rating'],
    errors='coerce'
)

df_cosmetics = df_cosmetics.dropna(
    subset=['rating']
)

df_cosmetics = df_cosmetics[
    (df_cosmetics['rating'] >= 0)
    &
    (df_cosmetics['rating'] <= 5)
]

# Limpieza de noofratings

df_cosmetics['noofratings'] = pd.to_numeric(
    df_cosmetics['noofratings'],
    errors='coerce'
).fillna(0)

print(df_cosmetics.dtypes)
print("\nDataset cargado correctamente.")
print(df_cosmetics.head())

product_name     object
website          object
country          object
category         object
subcategory      object
title_href       object
price           float64
brand            object
ingredients      object
form             object
type             object
color            object
size             object
rating          float64
noofratings     float64
dtype: object

Dataset cargado correctamente.
                                        product_name   website country  \
0  Carlton London Incense Eau da parfum, Premium ...  Flipkart   India   
1  CHARLENE SPRAY MIST PERFUME 30 - INTIMATE (PAC...  Flipkart   India   
2  CHARLENE SPRAY MIST PERFUME 30 - INTIMATE (PAC...  Flipkart   India   
3  DENVER Black Code Perfume - 60 | Eau de Parfum...  Flipkart   India   
4  Denver Hamilton Perfume - 100 | Long Lasting P...  Flipkart   India   

  category subcategory                                         title_href  \
0     body     perfume  https://www.amazon.in/Carlton-London-Limited-I..

### Celda 4: Crear el CORPUS

In [60]:
documentos = []

for _, fila in df_cosmetics.iterrows():

    texto = f"""
    Producto: {fila['product_name']}
    Marca: {fila['brand']}
    Categoría: {fila['category']}
    Precio: {fila['price']}
    Rating: {fila['rating']}
    Número de ratings: {fila['noofratings']}
    """

    documentos.append(
        Document(page_content=texto)
    )

print("Documentos creados:", len(documentos))

Documentos creados: 10483


### Celda 5: Crear Embeddings

In [50]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(
    documentos,
    embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k":5}
)

print("Base vectorial creada.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Base vectorial creada.


CELDA 6: Función RAG

In [71]:
chat_history = []

def preguntar_agente(pregunta):

    p = pregunta.lower()

    # CONSULTAS ESTADÍSTICAS CON PANDAS

    if "precio promedio" in p:

        respuesta = (
            f"El precio promedio de los productos es "
            f"{df_cosmetics['price'].mean():.2f}."
        )

    elif (
        "mejores calificaciones" in p
        or "mejor calificación" in p
        or "rating promedio" in p
        or "mejores marcas" in p
    ):

        top = (
            df_cosmetics
            .groupby("brand")["rating"]
            .mean()
            .sort_values(ascending=False)
            .head(5)
        )

        respuesta = (
            "Las marcas con mejores calificaciones promedio son:\n\n"
            + top.to_string()
        )

    elif "categorías" in p or "categorias" in p:

        categorias = (
            df_cosmetics["category"]
            .dropna()
            .unique()
        )

        respuesta = (
            f"Existen {len(categorias)} categorías:\n\n"
            + ", ".join(map(str, categorias))
        )

    elif (
        "más común" in p
        or "mas comun" in p
        or "categoría más común" in p
        or "categoria mas comun" in p
    ):

        top = (
            df_cosmetics["category"]
            .value_counts()
            .head(1)
        )

        respuesta = (
            f"La categoría más común es "
            f"{top.index[0]} "
            f"con {top.iloc[0]} productos."
        )

    elif (
        "superior a 4.5" in p
        or "mayor a 4.5" in p
    ):

        productos = (
            df_cosmetics[
                df_cosmetics["rating"] > 4.5
            ][["product_name", "brand", "rating"]]
            .head(10)
        )

        respuesta = (
            "Algunos productos con rating superior a 4.5 son:\n\n"
            + productos.to_string(index=False)
        )

    # CONSULTAS ABIERTAS CON RAG

    else:

        docs = retriever.invoke(pregunta)

        contexto = "\n".join(
            [doc.page_content for doc in docs[:2]]
        )

        prompt = f"""
Eres un asistente experto en cosméticos.

Utiliza únicamente la información del contexto.

Contexto:
{contexto}

Pregunta:
{pregunta}

Responde en español de forma breve y clara.
"""

        respuesta = llm.invoke(prompt)

    chat_history.append(
        (pregunta, respuesta)
    )

    return respuesta

CELDA 7: PRUEBAS AUTOMÁTICAS

In [73]:
print("----- PRUEBAS AUTOMÁTICAS -----")

preguntas = [
    "¿Cuál es el precio promedio de los productos?",
    "¿Qué marcas tienen mejores calificaciones?",
    "¿Qué categorías de cosméticos existen?",
    "¿Cuál es la categoría más común en el dataset?",
    "¿Qué productos tienen rating superior a 4.5?"
]

for p in preguntas:

    print("\n" + "="*70)
    print("PREGUNTA:")
    print(p)

    try:
        respuesta = preguntar_agente(p)

        print("\nRESPUESTA:")
        print(respuesta)

    except Exception as e:
        print("\nERROR:")
        print(e)

----- PRUEBAS AUTOMÁTICAS -----

PREGUNTA:
¿Cuál es el precio promedio de los productos?

RESPUESTA:
El precio promedio de los productos es 2453.53.

PREGUNTA:
¿Qué marcas tienen mejores calificaciones?

RESPUESTA:
Las marcas con mejores calificaciones promedio son:

brand
ADAMAI          5.0
KINDED          5.0
Krayons         5.0
ROSEN           5.0
JOTKAPARKASH    5.0

PREGUNTA:
¿Qué categorías de cosméticos existen?

RESPUESTA:
Existen 6 categorías:

body, eyes, hair, lips, skincare, face

PREGUNTA:
¿Cuál es la categoría más común en el dataset?

RESPUESTA:
La categoría más común es body con 2313 productos.

PREGUNTA:
¿Qué productos tienen rating superior a 4.5?

RESPUESTA:
Algunos productos con rating superior a 4.5 son:

                                        product_name           brand  rating
                      Deadsea Mud Purifying Mud Soap           Ahava     4.7
                       Sea-Kissed Mineral Shower Gel           Ahava     4.7
                   Mineral Botan

CELDA 8: CHAT INTERACTIVO

In [75]:
print("\n----- CHAT INTERACTIVO -----")
print("Escribe 'salir' para finalizar.\n")

while True:

    pregunta = input("Tú: ")

    if pregunta.lower() == "salir":
        print("Agente: ¡Hasta pronto!")
        break

    try:
        respuesta = preguntar_agente(pregunta)

        print("\nAgente:")
        print(respuesta)
        print()

    except Exception as e:
        print("\nError:")
        print(e)


----- CHAT INTERACTIVO -----
Escribe 'salir' para finalizar.

Tú: ¿Existen productos con un rating superior a 4.5?

Agente:
Algunos productos con rating superior a 4.5 son:

                                        product_name           brand  rating
                      Deadsea Mud Purifying Mud Soap           Ahava     4.7
                       Sea-Kissed Mineral Shower Gel           Ahava     4.7
                   Mineral Botanic Velvet Cream Wash           Ahava     4.7
                         24-Hour Deodorant Body Wash   American Crew     4.7
  3-in-1 Tea Tree Shampoo, Conditioner and Body Wash   American Crew     4.7
 Oatmilk Calendula Moisturizing Bubble Bath and Wash Babo Botanicals     4.7
3-in-1 Eucalyptus Remedy Bubble Bath, Wash & Shampoo Babo Botanicals     4.8
                 Calming Bubble Bath, Shampoo & Wash Babo Botanicals     4.7
            Travel Size Body Wash for Everyone Pouch           Beast     4.7
                       Castile All-In-One Soap Pouch   